# Exp11.0 — D0/D1 A/B/C/D context × fusion factorial

Aggregation-only notebook for `d0_d1_l1mem2_context_fusion_factorial_v3`. The six concrete architectures map to A, B-diag, B-dense, C, D-diag, and D-dense.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    p = Path.cwd() if start is None else Path(start)
    for candidate in (p, *p.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('repo root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_11_0_rsnn_history_internalization' / 'd0_d1_l1mem2_context_fusion_factorial_v3'
runs = pd.read_csv(root / 'run_metrics.csv')
methods = pd.read_csv(root / 'method_summary.csv')
contrasts = pd.read_csv(root / 'paired_contrast_summary.csv')
interactions = pd.read_csv(root / 'interaction_summary.csv')
gaps = pd.read_csv(root / 'temporal_gap_summary.csv')
activity = pd.read_csv(root / 'activity_summary.csv')
d0_sources = pd.read_csv(root / 'd0_source_metrics.csv')
runs


## Architecture map

A = FF/no Fusion; B = recurrent/no Fusion; C = FF/Fusion; D = recurrent/Fusion.


In [ ]:
architecture_map = runs[['architecture_case','topology','fusion']].drop_duplicates().sort_values(['architecture_case','topology'])
architecture_map


## Native performance


In [ ]:
cols = ['variant','l1_init','architecture_case','topology','fusion','seed','native_test_ba','window_test_ba','output_lif_test_ba','readout_comm_valid_whole_ba','readout_comm_valid_fixed250_ba','readout_comm_valid_temporal_gap']
runs[cols].sort_values(['variant','l1_init','architecture_case','topology','seed'])


## Fusion effect
C-A and D-B are emitted as `fusion_on_minus_off` for the matching topology.


In [ ]:
fusion_effect = contrasts[contrasts['contrast'] == 'fusion_on_minus_off'].copy()
fusion_effect


## Recurrence effect
B-A is diagonal/dense minus FF with Fusion off; D-C is the same contrast with Fusion on.


In [ ]:
recurrence_effect = contrasts[contrasts['contrast'].isin(['diagonal_minus_ff','dense_minus_ff'])].copy()
recurrence_effect


## Recurrence × Fusion interaction
The key interaction is `(D-C) - (B-A)`.


In [ ]:
recurrence_x_fusion = interactions[interactions['interaction'] == 'recurrence_x_fusion'].copy()
recurrence_x_fusion


## Paired D1-D0 effect


In [ ]:
d1_minus_d0 = contrasts[contrasts['contrast'] == 'd1_minus_d0'].copy()
d1_minus_d0


## Temporal internalization
Track Fixed250-minus-whole from L1 to context/RSNN to the final readout representation.


In [ ]:
gap_view = gaps[['variant','l1_init','architecture_case','topology','fusion','layer','support','fixed250_ba_mean','whole_ba_mean','temporal_gap_mean']]
gap_view.sort_values(['variant','l1_init','architecture_case','topology','support','layer'])


In [ ]:
plot_data = gaps[gaps['support'] == 'valid'].copy()
for (variant, l1_init, topology, fusion), group in plot_data.groupby(['variant','l1_init','topology','fusion']):
    ordered = group.set_index('layer').loc[['l1','rsnn','readout']].reset_index()
    plt.figure()
    plt.plot(ordered['layer'], ordered['temporal_gap_mean'], marker='o')
    plt.axhline(0, linewidth=1)
    plt.ylabel('Fixed250 BA - Whole BA')
    plt.title(f'{variant} / {l1_init} / {topology} / fusion={fusion}')
    plt.show()


## D0 matched-source sanity check


In [ ]:
d0_sources


## Recurrent activity diagnostics


In [ ]:
runs[['variant','l1_init','architecture_case','topology','fusion','seed','mean_abs_external_input','mean_abs_recurrent_input','recurrent_to_external_abs_ratio','recurrent_weight_norm']].sort_values(['variant','l1_init','architecture_case','topology','seed'])
